## Imports

In [0]:
from pyspark.sql.functions import row_number, count, max
from botocore.exceptions import NoCredentialsError
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from datetime import datetime
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


## Read the Perimeter and Businessdate

In [0]:
env = get_env_prefix()

In [0]:
run_date = dbutils.widgets.get("run_date")

run_date = spark.sql(f"""                    
                            select 
                                case 
                                    when try_cast('{run_date}' as date) is not null then try_cast('{run_date}' as date)
                                    else PreviousBusinessDay
                                end as BusinessDate
                            from `{env}-cdh`.reference.ref_calendar
                            where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                        """).first()[0]

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
# Read the table into a Pandas DataFrame
Perimeter_path = f"`{env}-reportinputfiles`.internal.EuriborFreezePerimeter"
Perimeter = spark.read.table(Perimeter_path)
Perimeter.createOrReplaceTempView("Perimeter")

## Read Tables

In [0]:
Portfolio = spark.sql(f"""                    
                        select distinct PortfolioName, PortfolioId, Prefix
                        from `{env}-cdh`.primary.Portfolio
                        where BusinessDate = '{run_date}'
                        """)

Portfolio.createOrReplaceTempView("Portfolio")

In [0]:
DExposure = spark.sql(f"""                    
                        select distinct BusinessDate, AgreementKey, Exposurekey, AssetSegment, PortfolioName, IsActiveFlag
                        from `{env}-cdh`.derivative.get_all_exposures_by_business_date('{run_date}')
                        inner join perimeter per
                        on per.CONTRACT_NUM = AgreementKey
                        and per.ACCOUNT_NUMBER = ExposureKey
                        """)

DExposure.createOrReplaceTempView("DExposure")

In [0]:
AllExposure = spark.sql(f"""                    
                            select BusinessDate, Exposurekey, CDI, CaseCode, PartyRelationshipType, PartyName, PartyTaxRegistrationNumber, SourceSystem
                            from `{env}-cdh`.primary.exposure
                        """)

AllExposure.createOrReplaceTempView("AllExposure")

In [0]:
Exposure_CaseCode = spark.sql(f"""                    
                            select Exposurekey, CDI, CaseCode, max(BusinessDate) over (partition by Exposurekey) AS Max_BusinessDate
                            from AllExposure
                            where SourceSystem = 'qca'
                        """)

Exposure_CaseCode.createOrReplaceTempView("Exposure_CaseCode")

In [0]:
exposure = spark.sql(f"""                    
                    select distinct 
                          dex.BusinessDate
                        , dex.Exposurekey
                        , dex.PortfolioName
                        , ex.CDI
                        , ex.PartyRelationshipType
                        , ex.PartyName
                        , ex.PartyTaxRegistrationNumber
                        , c.CaseCode AS QC_CaseCode
                    from DExposure dex
                    inner join AllExposure ex
                    on ex.BusinessDate = dex.BusinessDate
                    and ex.Exposurekey = dex.Exposurekey
                    left join Exposure_CaseCode c
                    on c.Exposurekey = substring(dex.Exposurekey, 4, LEN(dex.Exposurekey))
                    and c.CDI = ex.CDI
                    """)

exposure.createOrReplaceTempView("exposure")

In [0]:
DParty = spark.sql(f"""                    
                    select distinct 
                          ex.BusinessDate
                        , ex.CDI
                        , ex.PortfolioName
                        , ex.PartyRelationshipType
                        , PartyType
                        , ex.PartyName
                        , FirstNameGR
                        , LastNameGR
                        , FirstFatherNameGR
                        , TaxRegistrationNumber
                        , DeceasedFlag
                    from `{env}-cdh`.derivative.dparty dp
                    inner join exposure ex
                    on dp.BusinessDate = ex.BusinessDate
                    and ex.CDI = dp.CDI
                    """)

DParty.createOrReplaceTempView("DParty")

In [0]:
ContactDetails = spark.sql(f"""                    
                            select distinct 
                                  c.CDI
                                , Street
                                , PostalCode
                                , City
                                , County
                                , Country
                            from `{env}-cdh`.primary.contactdetails c
                            inner join DParty dp
                            on Sourcesystem = 'profits'
                            and CommunicationFlag = 1
                            and c.CDI = dp.CDI
                            and dp.BusinessDate = c.Businessdate
                            """)

ContactDetails.createOrReplaceTempView("ContactDetails")

## Create FreezeF

In [0]:
freezeF = spark.sql(f"""                    
                        select distinct
                                  dex.AgreementKey AS AgreementNumber
                                , dex.ExposureKey  AS Account
                                , ex.CDI
                                , dex.AssetSegment as BusinessUnit
                                , dex.PortfolioName
                                , SUBSTRING(dex.AgreementKey,4,LEN(dex.AgreementKey)) AS AgreementNumber_W
                                , split(substring(dex.AgreementKey, 4, len(dex.AgreementKey)), '-')[0] AS ContractNumberPhysical
                                , CASE 
                                    WHEN size(split(dex.AgreementKey, '-')) > 1 THEN split(dex.AgreementKey, '-')[1]
                                    ELSE split(substring(dex.AgreementKey, 4, len(dex.AgreementKey)), '-')[0]
                                  END AS ContractNumberCepal
                                , '' AS FreezeImplementationDate
                                , IsActiveFlag
                                , ex.QC_CaseCode
                                , PartyRelationshipType CustomerRelation
                        from exposure ex
                        inner join DExposure dex
                        on ex.Exposurekey = dex.Exposurekey
                    """)

freezeF.createOrReplaceTempView("freezeF")

## Create FreezeCDI 

In [0]:
freezeCDI = spark.sql(f"""                    
                        select distinct
                                dp.CDI
                              , coalesce(dp.TaxRegistrationNumber, ex.PartyTaxRegistrationNumber) AS TaxRegistrationNumber
                              , dp.PartyName
                              , CASE 
                                    WHEN PartyType = 'Individual' THEN FirstNameGR
                                    ELSE ''
                                END AS FirstName
                              , CASE 
                                    WHEN PartyType = 'Individual' THEN LastNameGR
                                    ELSE ''
                                END AS LastName
                              , CASE 
                                    WHEN PartyType = 'Individual' THEN FirstFatherNameGR
                                    ELSE ''
                                END AS FatherName
                              , PartyType
                              , DeceasedFlag
                              , Street
                              , PostalCode
                              , CONCAT(City,'-',County) AS City_County
                              , Country
                              , dp.PartyRelationshipType AS CustomerRelation
                              , dp.PortfolioName
                        from DParty dp
                        left join exposure ex
                        on dp.CDI = ex.CDI
                        left join ContactDetails cd
                        on dp.CDI = cd.CDI
                      """)

freezeCDI.createOrReplaceTempView("freezeCDI")

## Find and Pivot Multiple Accounts on same CDI

In [0]:
Multi = spark.sql(f"""                    
                    select distinct
                          CDI AS CDI_pivot
                        , PortfolioName AS PortfolioName_pivot
                        , AgreementNumber_W AS ApplicationNumber
                        , ContractNumberPhysical
                        , ContractNumberCepal
                        , FreezeImplementationDate
                        , BusinessUnit
                    from freezeF
                    where PortfolioName in ('Orion X Securitisation DAC', 'Galaxy II Funding DAC', 'Cosmos Retail')
                    """)

Multi.createOrReplaceTempView("Multi")

In [0]:
Pivoted_columns = ['ApplicationNumber', 'ContractNumberPhysical', 'ContractNumberCepal', 'FreezeImplementationDate', 'BusinessUnit']
Fixed_columns = ['CDI_pivot', 'PortfolioName_pivot']

# Add a column for the group index (to help with reshaping)
window_spec = Window.partitionBy('CDI_pivot', 'PortfolioName_pivot').orderBy('ApplicationNumber')
Multi = Multi.withColumn('index', row_number().over(window_spec))

# Convert to Pandas DataFrame for reshaping
Multi_pd = Multi.toPandas()

# Reshape the dataframe
reshaped_df = pd.pivot_table(Multi_pd, 
                             index=Fixed_columns, 
                             columns='index', 
                             values=Pivoted_columns, 
                             aggfunc='first')

# Flatten the MultiIndex in columns
reshaped_df.columns = [f"{col[0]}_{col[1]}" for col in reshaped_df.columns]

# Reset index to make the table readable
reshaped_df = reshaped_df.reset_index()

In [0]:
max_rnk = Multi.groupBy("CDI_pivot", "PortfolioName_pivot") \
               .agg(count("ApplicationNumber").alias("ApplicationCount")) \
               .agg(max("ApplicationCount")).collect()[0][0]

In [0]:
pivoted_columns_order = [f'{col}_{i}' for i in range(1, max_rnk + 1) for col in Pivoted_columns]

final_column_order = Fixed_columns + pivoted_columns_order

# Reorder the columns in the final DataFrame
reshaped_df = reshaped_df[final_column_order]

# Convert Pandas DataFrame to Spark DataFrame
reshaped_Multi = spark.createDataFrame(reshaped_df)

reshaped_Multi.createOrReplaceTempView("reshaped_Multi")

## Create the Export Files

In [0]:
File1 = spark.sql(f"""                    
                      select distinct
                            c.CDI AS CIF
                          , c.TaxRegistrationNumber AS AFM
                          , c.PartyName 
                          , c.FirstName 
                          , c.LastName  
                          , c.FatherName
                          , c.PartyType 
                          , c.Street AS 
                          , c.PostalCode
                          , c.City_County
                          , c.Country
                          , c.DeceasedFlag
                          , c.PortfolioName
                          , CASE
                                WHEN p.PortfolioName = 'N/A' OR p.PortfolioName IS NULL THEN 'N/A'
                                ELSE p.Prefix
                            END AS Prefix
                          , reshaped_Multi.*
                      from freezeCDI c
                      left join Portfolio p
                      on c.PortfolioName = p.PortfolioName
                      left join reshaped_Multi
                      on c.CDI = reshaped_Multi.CDI_pivot
                      and c.PortfolioName = reshaped_Multi.PortfolioName_pivot
                  """)

File1 = File1.drop('CDI_pivot', 'PortfolioName_pivot')
File1.createOrReplaceTempView("File1")

In [0]:
File2 = spark.sql(f"""                    
                      select distinct 
                              CDI
                            , Account
                            , p.PortfolioId
                            , AgreementNumber_W AS ContractNumber
                            , QC_CaseCode
                            , CustomerRelation
                      from freezeF
                      left join Portfolio p
                      on freezeF.PortfolioName = p.PortfolioName
                  """)

File2.createOrReplaceTempView("File2")

## Export to Volume and Table

In [0]:
# notebook: {env}-cdh-delivery
# volume: {env}-cdh-delivery.internalreports.euriborfreeze
# export path: \\dwhsideapps\Share\in\Prod\InternalReports\euriborfreeze

In [0]:
%pip install openpyxl

In [0]:
run_date_str = datetime.now().strftime("%Y%m%d")

#Define the output directory path
output_dir = f"{report_output}{run_date_str}/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_filename1 = f"{output_file_prefix}_File1_{run_date.strftime('%Y%m%d')}.xlsx"
output_path1 = os.path.join(output_dir, output_filename1)

output_filename2 = f"{output_file_prefix}_File2_{run_date.strftime('%Y%m%d')}.xlsx"
output_path2 = os.path.join(output_dir, output_filename2)

In [0]:
# File 1
# Create an Excel writer object
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
    # Save the subset to a temporary CSV file
    temp_file = os.path.join(output_dir, f"File1.csv")
    File1.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
    
    # Read the CSV back into Pandas
    csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
    if csv_files:
        pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]))
        
        # Write to an Excel sheet
        pandas_df.to_excel(writer, index=False)
    for file in os.listdir(temp_file):
        os.remove(os.path.join(temp_file, file))

with open(output_path1, "wb") as f:
    f.write(output_buffer.getvalue())

In [0]:
# File 2
# Create an Excel writer object
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
    # Save the subset to a temporary CSV file
    temp_file = os.path.join(output_dir, f"File2.csv")
    File2.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
    
    # Read the CSV back into Pandas
    csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
    if csv_files:
        pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]))
        
        # Write to an Excel sheet
        pandas_df.to_excel(writer, index=False)
    for file in os.listdir(temp_file):
        os.remove(os.path.join(temp_file, file))

with open(output_path2, "wb") as f:
    f.write(output_buffer.getvalue())